# 4.1 Model validation and baseline characterization

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

def _resolve_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('config.py not found in cwd or parents')

PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/markus/in-silico-vg-analysis


In [2]:
from config import (
    GENE_PATHS,
    VARIANT_PATHS,
    SOURCE_PALETTE,
    DISPLAY_NAMES,
    DISPLAY_NAMES_SHORT,
)
from utils.plot_utils import save_plot, autosave

NOTEBOOK_NAME = 'final_figure'

print('\nDatasets:')
for name in ['clingen', 'clingen_null', 'background', 'background_null']:
    print(f'  {name}: {GENE_PATHS[name].name}')


Datasets:
  clingen: ClinGen_HI_Gnomad_genes_21022026.parquet
  clingen_null: ClinGen_HI_Synth_genes_21022026.parquet
  background: Background_Gnomad_genes_21022026.parquet
  background_null: Background_Synth_genes_21022026.parquet


In [3]:
real_gene_dfs = []
synth_gene_dfs = []

for key, path in GENE_PATHS.items():
    lf = pl.scan_parquet(path).with_columns(pl.lit(key).alias('source'))
    if 'null' in key:
        synth_gene_dfs.append(lf)
    else:
        real_gene_dfs.append(lf)

df_real = pl.concat(real_gene_dfs).collect()
df_synth = pl.concat(synth_gene_dfs).collect()

df_hi = df_real.filter(pl.col('source') == 'clingen')
df_bg = df_real.filter(pl.col('source') == 'background')
df_hi_synth = df_synth.filter(pl.col('source') == 'clingen_null')
df_bg_synth = df_synth.filter(pl.col('source') == 'background_null')

print(f'ClinGen HI: {df_hi.height} genes')
print(f'Background: {df_bg.height} genes')
print(f'ClinGen HI (Synth): {df_hi_synth.height} genes')
print(f'Background (Synth): {df_bg_synth.height} genes')

ClinGen HI: 316 genes
Background: 349 genes
ClinGen HI (Synth): 316 genes
Background (Synth): 349 genes


In [4]:
real_variant_dfs = []
synth_variant_dfs = []

for key, path in VARIANT_PATHS.items():
    lf = pl.scan_parquet(path).with_columns(pl.lit(key).alias('source'))
    if 'null' in key:
        synth_variant_dfs.append(lf)
    else:
        real_variant_dfs.append(lf)

df_real = pl.concat(real_variant_dfs).collect()
df_synth = pl.concat(synth_variant_dfs).collect()

df_hi_variants = df_real.filter(pl.col('source') == 'clingen')
df_bg_variants = df_real.filter(pl.col('source') == 'background')
df_hi_synth_variants = df_synth.filter(pl.col('source') == 'clingen_null')
df_bg_synth_variants = df_synth.filter(pl.col('source') == 'background_null')

print(f'ClinGen HI: {df_hi_variants.height} variants')
print(f'Background: {df_bg_variants.height} variants')
print(f'ClinGen HI (Synth): {df_hi_synth_variants.height} variants')
print(f'Background (Synth): {df_bg_synth_variants.height} variants')

ClinGen HI: 1743183 variants
Background: 1999142 variants
ClinGen HI (Synth): 1556962 variants
Background (Synth): 1788286 variants


## 4.1.1 Variant-level effect distributions

In [ ]:
# TODO: 90% of cis-SNPs show near-zero effects (|Δᵢ| < 0.02)
#Heavy-tailed: gene-level variance driven by handful of impactful SNPs
#N₉₀ metric: ClinGen genes N₉₀ ≈ 1–4; Background N₉₀ ≈ 30–60
# -> Figure 1: Variant-level effect scatter (AF vs Δᵢ, bubble = VARᵢ) for representative ClinGen vs Background gene

## 4.1.2 AlphaGenome tissue-specific correlation

In [ ]:
# Positive Spearman ρ in ~80% of GTEx tissues (peak ρ ≈ 0.60, ovary); ~20% negative (e.g. liver ρ ≈ −0.20)
# Borzoi exon-mask: far fewer positive tissues, peak ρ ≈ 0.35
# AlphaGenome chosen as primary model
# -> Figure 2: Tissue-specific Spearman ρ barplot (AG vs Borzoi)
# Slide sources: pp. 61-80 (tissue correlation), pp. 180 (predicted vs observed Vg)


## 4.1.3 eQTL concordance


In [ ]:
# vg_predicted vs vg_eqtl: Pearson 0.25–0.28; sum_sq_raw_score vs vg_eqtl: 0.44–0.47
# Validates that DL predictions capture real regulatory signal, though imperfectly
# Slide sources: pp. 121-140 (eQTL validation heatmaps)
